# CAP Sleep Database: download and integrity check

Downloads the 108 records of the CAP Sleep Database (PhysioNet) and their annotation files to `sleep_project/cap_edf` and `sleep_project/cap_annotations`, lists the available channels and verifies every file with SHA-256.

In [ ]:
# Google Drive, CAP record list and annotation files
from google.colab import drive
drive.mount('/content/drive')

import re, requests, pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

S3   = "https://physionet-open.s3.amazonaws.com/capslpdb/1.0.0/"
ROOT = Path("/content/drive/MyDrive/sleep_project")
EDF  = ROOT / "cap_edf";         EDF.mkdir(parents=True, exist_ok=True)
ANN  = ROOT / "cap_annotations"; ANN.mkdir(parents=True, exist_ok=True)

records = [r.strip().replace(".edf", "") for r in requests.get(S3 + "RECORDS", timeout=60).text.split()]
for f in ["SHA256SUMS.txt", "gender-age.xlsx"] + [f"{r}.txt" for r in records]:
    if not (ANN / f).exists():
        (ANN / f).write_bytes(requests.get(S3 + f, timeout=60).content)
print(len(records), "records |", len(list(ANN.glob("*.txt"))), "annotation files")
!df -h /content/drive

Mounted at /content/drive
108 records | 109 annotation files
Filesystem      Size  Used Avail Use% Mounted on
drive           236G   53G  184G  23% /content/drive


In [ ]:
# Channel inventory: records with EEG, chin EMG and ECG
def header(rec):
    b = requests.get(S3 + f"{rec}.edf", headers={"Range": "bytes=0-65535"}, timeout=60)
    total = int(b.headers["Content-Range"].split("/")[-1]); b = b.content
    ns = int(b[252:256].decode().strip())
    labels = [b[256 + i*16: 256 + (i+1)*16].decode("latin-1").strip() for i in range(ns)]
    return rec, labels, total

with ThreadPoolExecutor(16) as ex:
    hdrs = list(ex.map(header, records))

norm = lambda s: re.sub(r"[^A-Z0-9]", "", s.upper())
rows = []
for rec, labels, size in hdrs:
    N = [norm(l) for l in labels]
    rows.append({"record": rec, "group": re.match(r"[a-z]+", rec).group(), "size_GB": size / 1e9,
                 "EEG": any(x in ("C4A1", "C3A2", "C4M1", "C3M2") or x.startswith(("C4", "C3")) for x in N),
                 "EMG_chin": any(x in ("EMG1EMG2", "EMGEMG", "CHIN1", "CHIN", "EMG") or x.startswith(("CHIN", "EMG")) for x in N),
                 "ECG": any("ECG" in x or "EKG" in x for x in N),
                 "labels": "|".join(labels)})
inv = pd.DataFrame(rows)
inv["usable"] = inv.EEG & inv.EMG_chin & inv.ECG & (inv.record != "nfle27")
inv.to_csv(ROOT / "cap_inventory.csv", index=False)

print(inv.groupby("group")[["EEG", "EMG_chin", "ECG", "usable"]].sum().astype(int).to_string())
u = inv[inv.usable]
print(f"\nusable: {len(u)} records, {u.size_GB.sum():.1f} GB  (all 108: {inv.size_GB.sum():.1f} GB)")

       EEG  EMG_chin  ECG  usable
group                            
brux     2         2    2       2
ins      9         9    9       9
n       16        13   15      13
narco    5         5    5       5
nfle    40        40   40      39
plm     10        10   10      10
rbd     22        22   22      22
sdb      4         4    4       4

usable: 104 records, 42.2 GB  (all 108: 43.1 GB)


In [ ]:
# Download all EDF files (resumable, parallel)
import time, threading, requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

S3  = "https://physionet-open.s3.amazonaws.com/capslpdb/1.0.0/"
EDF = Path("/content/drive/MyDrive/sleep_project/cap_edf"); EDF.mkdir(parents=True, exist_ok=True)
records = [r.strip().replace(".edf", "") for r in requests.get(S3 + "RECORDS", timeout=60).text.split()]

def size_of(r):
    return r, int(requests.head(S3 + f"{r}.edf", timeout=60).headers["Content-Length"])
with ThreadPoolExecutor(16) as ex:
    sizes = dict(ex.map(size_of, records))
TOTAL = sum(sizes.values())

def local_size(r):
    p = EDF / f"{r}.edf"
    return p.stat().st_size if p.exists() else 0

stop = threading.Event()

def download(r):
    path, total = EDF / f"{r}.edf", sizes[r]
    for attempt in range(20):
        if stop.is_set(): return r, "stopped"
        have = local_size(r)
        if have == total: return r, "ok"
        if have > total:
            path.unlink(); have = 0
        try:
            with requests.get(S3 + f"{r}.edf", headers={"Range": f"bytes={have}-"},
                              stream=True, timeout=(15, 120)) as resp:
                resp.raise_for_status()
                with open(path, "ab") as f:
                    for chunk in resp.iter_content(1 << 22):
                        if stop.is_set(): return r, "stopped"
                        f.write(chunk)
        except Exception as e:
            time.sleep(min(60, 5 * (attempt + 1)))
    return r, "ok" if local_size(r) == total else "FAILED"

def monitor():
    t0, b0 = time.time(), sum(local_size(r) for r in records)
    last_t, last_b = t0, b0
    while not stop.wait(300):
        b = sum(local_size(r) for r in records)
        done = sum(local_size(r) == sizes[r] for r in records)
        now = time.time()
        speed = (b - last_b) / (now - last_t) / 1e6
        avg = (b - b0) / (now - t0) / 1e6
        eta = (TOTAL - b) / 1e6 / avg / 60 if avg > 0 else float("nan")
        print(f"[{time.strftime('%H:%M:%S')}] files {done}/{len(records)} | "
              f"{b/1e9:5.1f}/{TOTAL/1e9:.1f} GB ({100*b/TOTAL:4.1f}%) | "
              f"{speed:5.1f} MB/s now, {avg:5.1f} MB/s avg | ETA {eta:5.1f} min", flush=True)
        last_t, last_b = now, b

already = sum(local_size(r) == sizes[r] for r in records)
print(f"{already}/{len(records)} files already complete | "
      f"{sum(local_size(r) for r in records)/1e9:.1f}/{TOTAL/1e9:.1f} GB on Drive\n", flush=True)

mon = threading.Thread(target=monitor, daemon=True); mon.start()
results = {}
try:
    with ThreadPoolExecutor(6) as ex:
        futs = [ex.submit(download, r) for r in records if local_size(r) != sizes[r]]
        for fut in as_completed(futs):
            r, status = fut.result(); results[r] = status
            if status != "ok": print(f"  {r}: {status}", flush=True)
except KeyboardInterrupt:
    print("\nstopping... (partial files are kept; just run this cell again to resume)", flush=True)
finally:
    stop.set()

done = sum(local_size(r) == sizes[r] for r in records)
print(f"\nDONE: {done}/{len(records)} complete | failed: {[r for r, s in results.items() if s == 'FAILED']}")

32/108 files already complete | 10.2/43.1 GB on Drive


DONE: 108/108 complete | failed: []


In [ ]:
# SHA-256 check of the downloaded files
import hashlib
ANN = Path("/content/drive/MyDrive/sleep_project/cap_annotations")
sums = dict(l.split()[::-1] for l in (ANN / "SHA256SUMS.txt").read_text().splitlines() if l.strip())
bad = []
for i, r in enumerate(records, 1):
    h = hashlib.sha256()
    with open(EDF / f"{r}.edf", "rb") as f:
        for chunk in iter(lambda: f.read(1 << 24), b""): h.update(chunk)
    if h.hexdigest() != sums.get(f"{r}.edf"): bad.append(r)
    if i % 10 == 0: print(f"checked {i}/{len(records)}", flush=True)
print("BAD:", bad if bad else "none — all files OK")

checked 10/108
checked 20/108
checked 30/108
checked 40/108
checked 50/108
checked 60/108
checked 70/108
checked 80/108
checked 90/108
checked 100/108
BAD: none — all files OK


In [ ]:
# Save everything to Drive
from google.colab import drive
drive.flush_and_unmount()
print("all changes saved to Drive")

Drive not mounted, so nothing to flush and unmount.
all changes saved to Drive


In [ ]:
# Independent re-check after remounting Drive
from google.colab import drive
drive.mount('/content/drive')

import hashlib
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/sleep_project")
EDF, ANN = ROOT / "cap_edf", ROOT / "cap_annotations"

sums = dict(l.split()[::-1] for l in (ANN / "SHA256SUMS.txt").read_text().splitlines() if l.strip())
records = sorted(k[:-4] for k in sums if k.endswith(".edf"))
bad, missing = [], []
for i, r in enumerate(records, 1):
    p = EDF / f"{r}.edf"
    if not p.exists(): missing.append(r); continue
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 24), b""): h.update(chunk)
    if h.hexdigest() != sums[f"{r}.edf"]: bad.append(r)
    if i % 10 == 0: print(f"checked {i}/{len(records)}", flush=True)
print("\nmissing:", missing or "none")
print("BAD:", bad or "none — all files OK")

Mounted at /content/drive
checked 10/108
checked 20/108
checked 30/108
checked 40/108
checked 50/108
checked 60/108
checked 70/108
checked 80/108
checked 90/108
checked 100/108

missing: none
BAD: none — all files OK
